In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, ToolMessage, AIMessage, HumanMessage

In [2]:
@tool
def get_weather(location: str) -> str:
    """ 
    Gets weather in the given location
    """
    return "Sunny with 31C"

@tool
def get_population(location: str) -> str:
    """
    Gets population in the given location
    """
    return "1 million."

In [3]:
gemini = init_chat_model(
    "gemini-2.5-flash", model_provider="google_genai", temperature=0
)

messages = [
    SystemMessage(
        'You can use tools when necessary, but otherwise answer the question using your own knowledge.'
    ),
    #HumanMessage(content="What is the capital of India")
    HumanMessage(content="What is the weather in Vizag and its population")
]

gemini_with_tools = gemini.bind_tools([get_weather, get_population])

In [4]:
while True:
    ai_msg = gemini_with_tools.invoke(messages)
    # Add AI message to messages
    messages.append(ai_msg)

    # Terminate loop if no tool is to be called
    if len(ai_msg.tool_calls) == 0:
        break

    # call tool selected by LLM
    for tool_call in ai_msg.tool_calls:
        selected_tool = eval(tool_call["name"])
        tool_result = selected_tool.invoke(tool_call)
        messages.append(tool_result)


In [5]:
for message in messages:
    message.pretty_print()

================================ System Message ================================

You can use tools when necessary, but otherwise answer the question using your own knowledge.
================================ Human Message =================================

What is the weather in Vizag and its population
================================== Ai Message ==================================
Tool Calls:
  get_weather (cbd6d240-15b8-4069-95fa-65099189d181)
 Call ID: cbd6d240-15b8-4069-95fa-65099189d181
  Args:
    location: Vizag
  get_population (ad23b5e9-49a2-4085-bb95-a39fa31cc51c)
 Call ID: ad23b5e9-49a2-4085-bb95-a39fa31cc51c
  Args:
    location: Vizag
================================= Tool Message =================================
Name: get_weather

Sunny with 31C
================================= Tool Message =================================
Name: get_population

1 million.
================================== Ai Message ==================================

The weather in Vizag is Sunny w